# Task 5: SQL Advanced (CTEs & Window Functions)

In [1]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("northstar_python.db")
cur = conn.cursor()

# Confirms the tables exist with the expected row counts before running
# anything else.
print("orders:", cur.execute("SELECT COUNT(*) FROM orders").fetchone()[0])
print("regions:", cur.execute("SELECT COUNT(*) FROM regions").fetchone()[0])

orders: 999
regions: 8


In [2]:
# Runs a SQL string and returns the result as a readable table instead
# of raw tuples.
def q(sql):
    return pd.read_sql_query(sql, conn)

## 1. CTE vs subquery

In [3]:
# CTE version: computes each customer's total per region, then each
# region's total, then uses ROW_NUMBER to pick the single top customer
# per region before dividing their revenue by the region total.
q("""
WITH customer_totals AS (
    SELECT region, customer_id, SUM(total_amount) AS customer_revenue
    FROM orders
    GROUP BY region, customer_id
),
region_totals AS (
    SELECT region, SUM(total_amount) AS region_revenue
    FROM orders
    GROUP BY region
),
top_customer_per_region AS (
    SELECT region, customer_id, customer_revenue,
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY customer_revenue DESC) AS rn
    FROM customer_totals
)
SELECT t.region, t.customer_id,
       ROUND(t.customer_revenue, 2) AS top_customer_revenue,
       ROUND(r.region_revenue, 2) AS region_revenue,
       ROUND(100.0 * t.customer_revenue / r.region_revenue, 2) AS pct_of_region
FROM top_customer_per_region t
JOIN region_totals r ON t.region = r.region
WHERE t.rn = 1
ORDER BY pct_of_region DESC;
""")

,region,customer_id,top_customer_revenue,region_revenue,pct_of_region
0,Wales,CUST-0100,2314.71,35158.23,6.58
1,East,CUST-0155,3419.93,65605.20,5.21
2,North,CUST-0160,3854.04,89670.44,4.30
3,Midlands,CUST-0105,2395.81,57027.34,4.20
4,West,CUST-0010,2364.09,56225.58,4.20
5,Scotland,CUST-0089,2021.67,51617.42,3.92
6,South,CUST-0002,3265.89,84435.95,3.87
7,London,CUST-0075,2552.36,73878.89,3.45


In [4]:
# Subquery version: finds each customer's top spend per region by
# matching their total against the MAX customer total within that same
# region, computed inline via a correlated subquery instead of a named
# CTE step. The region total is also recomputed separately in a second
# subquery rather than reused from a named block.
q("""
SELECT
    ct.region, ct.customer_id,
    ROUND(ct.customer_revenue, 2) AS top_customer_revenue,
    ROUND((SELECT SUM(total_amount) FROM orders o2 WHERE o2.region = ct.region), 2) AS region_revenue,
    ROUND(100.0 * ct.customer_revenue /
        (SELECT SUM(total_amount) FROM orders o2 WHERE o2.region = ct.region), 2) AS pct_of_region
FROM (
    SELECT region, customer_id, SUM(total_amount) AS customer_revenue
    FROM orders
    GROUP BY region, customer_id
) ct
WHERE ct.customer_revenue = (
    SELECT MAX(customer_revenue) FROM (
        SELECT customer_id, SUM(total_amount) AS customer_revenue
        FROM orders o3 WHERE o3.region = ct.region
        GROUP BY customer_id
    )
)
ORDER BY pct_of_region DESC;
""")

,region,customer_id,top_customer_revenue,region_revenue,pct_of_region
0,Wales,CUST-0100,2314.71,35158.23,6.58
1,East,CUST-0155,3419.93,65605.20,5.21
2,North,CUST-0160,3854.04,89670.44,4.30
3,Midlands,CUST-0105,2395.81,57027.34,4.20
4,West,CUST-0010,2364.09,56225.58,4.20
5,Scotland,CUST-0089,2021.67,51617.42,3.92
6,South,CUST-0002,3265.89,84435.95,3.87
7,London,CUST-0075,2552.36,73878.89,3.45


**Read:** both agree, Wales's top customer contributes 6.58% of regional revenue, the highest of any region. The CTE version reads better since each named block states one idea, while the subquery repeats the same region-sum logic twice and relies on an equality match against MAX, which would silently return more than one row if two customers ever tied for the top spot.

## 2. Ranking windows

In [5]:
# Numbers each region's orders 1, 2, 3, ... in date order by resetting
# the count at the start of every new region (the PARTITION BY).
q("""
SELECT region, order_id, order_date,
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY order_date) AS order_seq
FROM orders
ORDER BY region, order_seq;
""")

,region,order_id,order_date,order_seq
0,East,NS-00381,2024-01-02,1
1,East,NS-00300,2024-01-08,2
2,East,NS-00540,2024-01-08,3
3,East,NS-00272,2024-01-09,4
4,East,NS-00138,2024-01-12,5
...,...,...,...,...
994,West,NS-00797,2024-12-19,121
995,West,NS-00417,2024-12-19,122
996,West,NS-00208,2024-12-19,123
997,West,NS-00914,2024-12-23,124


In [6]:
# Ranks customers by total spend within their own region. RANK() and
# DENSE_RANK() only produce different numbers where two customers tie
# on total_spend, which is what this query is built to expose.
q("""
WITH customer_totals AS (
    SELECT region, customer_id, SUM(total_amount) AS total_spend
    FROM orders
    GROUP BY region, customer_id
)
SELECT region, customer_id, ROUND(total_spend, 2) AS total_spend,
       RANK() OVER (PARTITION BY region ORDER BY total_spend DESC) AS rnk,
       DENSE_RANK() OVER (PARTITION BY region ORDER BY total_spend DESC) AS dense_rnk
FROM customer_totals
ORDER BY region, total_spend DESC;
""")

,region,customer_id,total_spend,rnk,dense_rnk
0,East,CUST-0155,3419.93,1,1
1,East,CUST-0043,2417.43,2,2
2,East,CUST-0141,2198.37,3,3
3,East,CUST-0289,1860.78,4,4
4,East,CUST-0158,1774.70,5,5
...,...,...,...,...,...
818,West,CUST-0002,48.41,101,101
819,West,CUST-0219,41.72,102,102
820,West,CUST-0085,35.82,103,103
821,West,CUST-0046,28.86,104,104


**Read:** London's two customers tied at £521.12 both get rank 8 under either function, but the next row gets RANK 10 (it skips two positions for the two tied rows) versus DENSE_RANK 9 (it never skips).

In [7]:
# Splits every customer into 4 equal-sized buckets by spend, ordered
# highest to lowest, so bucket 1 is the top-spending quarter of
# customers and bucket 4 is the lowest-spending quarter.
q("""
WITH customer_totals AS (
    SELECT customer_id, SUM(total_amount) AS total_spend
    FROM orders
    GROUP BY customer_id
)
SELECT customer_id, ROUND(total_spend, 2) AS total_spend,
       NTILE(4) OVER (ORDER BY total_spend DESC) AS spend_quartile
FROM customer_totals
ORDER BY total_spend DESC;
""")

,customer_id,total_spend,spend_quartile
0,CUST-0135,6260.59,1
1,CUST-0296,5866.81,1
2,CUST-0074,5299.69,1
3,CUST-0160,5266.72,1
4,CUST-0001,5226.45,1
...,...,...,...
285,CUST-0199,160.93,4
286,CUST-0236,129.47,4
287,CUST-0048,124.06,4
288,CUST-0062,84.89,4


**Read:** quartile 1 ranges £2,471.90 to £6,260.59; quartile 4 ranges £36.12 to £859.61, roughly a 7x gap.

## 3. Aggregating windows

In [8]:
# Adds up total_amount for each region as orders are processed in date
# order, so each row shows the cumulative revenue up to and including
# that order.
q("""
SELECT region, order_id, order_date, total_amount,
       ROUND(SUM(total_amount) OVER (
           PARTITION BY region ORDER BY order_date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ), 2) AS running_total
FROM orders
ORDER BY region, order_date;
""")

,region,order_id,order_date,total_amount,running_total
0,East,NS-00381,2024-01-02,111.68,111.68
1,East,NS-00300,2024-01-08,61.90,173.58
2,East,NS-00540,2024-01-08,621.10,794.68
3,East,NS-00272,2024-01-09,598.80,1393.48
4,East,NS-00138,2024-01-12,78.84,1472.32
...,...,...,...,...,...
994,West,NS-00797,2024-12-19,1174.32,54912.42
995,West,NS-00417,2024-12-19,474.13,55386.55
996,West,NS-00208,2024-12-19,115.12,55501.67
997,West,NS-00914,2024-12-23,133.45,55635.12


In [9]:
# Divides each order's value by the full-year total for its region,
# showing what share of that region's annual revenue any single order
# represents.
q("""
SELECT region, order_id, total_amount,
       ROUND(100.0 * total_amount / SUM(total_amount) OVER (PARTITION BY region), 3)
           AS pct_of_region_total
FROM orders
ORDER BY region, pct_of_region_total DESC;
""")

,region,order_id,total_amount,pct_of_region_total
0,East,NS-00243,1839.90,2.805
1,East,NS-00311,1774.70,2.705
2,East,NS-00924,1570.29,2.394
3,East,NS-00006,1553.30,2.368
4,East,NS-00296,1546.64,2.357
...,...,...,...,...
994,West,NS-00218,31.99,0.057
995,West,NS-00333,28.76,0.051
996,West,NS-00970,28.86,0.051
997,West,NS-00321,16.09,0.029


**Read:** Wales's largest single order (£1,865.00) alone makes up 5.31% of the region's entire annual revenue.

## 4. Lag/lead

In [10]:
# Aggregates revenue by region and month, then LAG() pulls in each
# month's own value from the row directly before it in the same
# region's sequence, which makes the month-over-month % calculation
# possible in one pass.
q("""
WITH monthly_region_revenue AS (
    SELECT region, strftime('%Y-%m', order_date) AS order_month,
           SUM(total_amount) AS revenue
    FROM orders
    GROUP BY region, order_month
)
SELECT region, order_month, ROUND(revenue, 2) AS revenue,
       ROUND(LAG(revenue) OVER (PARTITION BY region ORDER BY order_month), 2)
           AS prev_month_revenue,
       ROUND(
           100.0 * (revenue - LAG(revenue) OVER (PARTITION BY region ORDER BY order_month))
           / LAG(revenue) OVER (PARTITION BY region ORDER BY order_month),
       2) AS mom_growth_pct
FROM monthly_region_revenue
ORDER BY region, order_month;
""")

,region,order_month,revenue,prev_month_revenue,mom_growth_pct
0,East,2024-01,3794.15,NaN,NaN
1,East,2024-02,8677.91,3794.15,128.72
2,East,2024-03,4612.96,8677.91,-46.84
3,East,2024-04,6833.26,4612.96,48.13
4,East,2024-05,4641.71,6833.26,-32.07
...,...,...,...,...,...
91,West,2024-08,7409.13,3335.00,122.16
92,West,2024-09,3934.45,7409.13,-46.90
93,West,2024-10,4835.04,3934.45,22.89
94,West,2024-11,5513.24,4835.04,14.03


In [11]:
# Reuses the same growth calculation as above, then sorts every
# region-month by growth ascending and keeps only the single worst row.
worst = q("""
WITH monthly_region_revenue AS (
    SELECT region, strftime('%Y-%m', order_date) AS order_month,
           SUM(total_amount) AS revenue
    FROM orders
    GROUP BY region, order_month
),
with_growth AS (
    SELECT region, order_month, revenue,
           LAG(revenue) OVER (PARTITION BY region ORDER BY order_month) AS prev_month_revenue,
           100.0 * (revenue - LAG(revenue) OVER (PARTITION BY region ORDER BY order_month))
               / LAG(revenue) OVER (PARTITION BY region ORDER BY order_month) AS mom_growth_pct
    FROM monthly_region_revenue
)
SELECT region, order_month, ROUND(revenue, 2) AS revenue,
       ROUND(prev_month_revenue, 2) AS prev_month_revenue,
       ROUND(mom_growth_pct, 2) AS mom_growth_pct
FROM with_growth
WHERE mom_growth_pct IS NOT NULL
ORDER BY mom_growth_pct ASC
LIMIT 1;
""")
worst

,region,order_month,revenue,prev_month_revenue,mom_growth_pct
0,North,2024-07,2209.1,13661.03,-83.83


**Read:** the worst region-month is North, July 2024, at negative 83.83% (£13,661.03 down to £2,209.10). Individual regions swing harder than the whole business since they have far fewer orders per month to average out.

## 5. Multi-step CTE

In [12]:
# Step 1 sums spend and counts orders per customer per region. Step 2
# ranks customers inside their own region by that spend. The final
# SELECT then just filters down to rank 3 or better.
q("""
WITH customer_region_stats AS (
    SELECT region, customer_id,
           SUM(total_amount) AS total_spend,
           COUNT(*) AS order_count
    FROM orders
    GROUP BY region, customer_id
),
ranked_customers AS (
    SELECT region, customer_id, total_spend, order_count,
           RANK() OVER (PARTITION BY region ORDER BY total_spend DESC) AS spend_rank
    FROM customer_region_stats
)
SELECT region, spend_rank, customer_id,
       ROUND(total_spend, 2) AS total_spend, order_count
FROM ranked_customers
WHERE spend_rank <= 3
ORDER BY region, spend_rank;
""")

,region,spend_rank,customer_id,total_spend,order_count
0,East,1,CUST-0155,3419.93,3
1,East,2,CUST-0043,2417.43,2
2,East,3,CUST-0141,2198.37,2
3,London,1,CUST-0075,2552.36,2
4,London,2,CUST-0019,2187.35,4
5,London,3,CUST-0251,2092.04,2
6,Midlands,1,CUST-0105,2395.81,2
7,Midlands,2,CUST-0123,1830.00,2
8,Midlands,3,CUST-0234,1686.42,1
9,North,1,CUST-0160,3854.04,5


**Read:** 24 rows in total, 8 regions times 3 customers each. North's top customer, CUST-0160, leads with £3,854.04 across 5 orders.

## 6. Reflection

SQL over pandas: query 5 (top 3 customers per region by spend) is a
good fit for SQL because the database already has everything I need,
one query gives me the answer, and it will still be correct next
month without me having to re-run any Python.

pandas over SQL: query 4 told me North's revenue dropped 83.83% in
July 2024, but not why. To dig into that I'd rather use pandas, pull
North's July orders into a dataframe and plot them, since exploring
and visualizing data like that is easier in pandas than in SQL.